# HR DATA CLEANING
 **Project Objectives**
- This project transformed raw, unstructured HR data into an analysis-ready dataset by leveraging modular and reusable Python functions.

**Phase 1. Imported the Required Libraries and Loaded Dataset**

In [1]:
#Import required libraries
import pandas as pd
import numpy as np

#Load Dataset
hrdata= pd.read_excel(r"C:\Users\HP\OneDrive\Desktop\LuxDev Tutorials\Python Projects\HR Data Analysis Project.ipynb\HR_Dirty_Data.xlsx")

**Phase 2. Initial Data Quality Assessment Report**

In [2]:
quality_report = pd.DataFrame({
    "Column": hrdata.columns,
    "Total Columns": len(hrdata.columns),
    "Data_Type": hrdata.dtypes.astype(str),
    "Total_Rows": len(hrdata),
    "Missing_Values": hrdata.isna().sum().values,
    "Missing_Percentage": (hrdata.isna().mean() * 100).round(2).values,
    "Unique_Values": hrdata.nunique().values,
})

quality_report


,Column,Total Columns,Data_Type,Total_Rows,Missing_Values,Missing_Percentage,Unique_Values
Employee ID,Employee ID,21,object,876,7,0.80,852
First Name,First Name,21,str,876,11,1.26,123
Last Name,Last Name,21,str,876,9,1.03,84
Department,Department,21,str,876,12,1.37,22
Salary,Salary,21,object,876,9,1.03,855
Hire Date,Hire Date,21,object,876,12,1.37,769
Age,Age,21,object,876,7,0.80,59
Gender,Gender,21,str,876,20,2.28,9
Performance Score,Performance Score,21,object,876,15,1.71,19
Full-Time,Full-Time,21,str,876,9,1.03,10


**Phase 3. Data Cleaning Steps Employed**

**Step 1: Rename  and Standardize Column Heads**

In [3]:
print(f"Current Column Names Heading\n",hrdata.columns)

Current Column Names Heading
 Index(['Employee ID', 'First Name', 'Last Name', 'Department', 'Salary',
       'Hire Date', 'Age', 'Gender', 'Performance Score', 'Full-Time', 'Bonus',
       'Marital Status', 'Education Level', 'Work Experience (Years)',
       'Employee Type', 'Office Location', 'Project Count',
       'Last Promotion Year', 'Remote Work Status', 'Annual Training Hours',
       'Manager Feedback Score'],
      dtype='str')


In [4]:
#Method 1. Column by column  renaming
hrdata = hrdata.rename(columns={'Employee ID': 'employee_id'})
hrdata = hrdata.rename(columns={'First Name': 'first_name'})
hrdata = hrdata.rename(columns={'Last Name': 'last_name'})

In [5]:
#Method 2. Created a function to rename heads
def clean_column_names(hrdata):
    """Converts all column headers to lowercase, replaces spaces with underscores, 
    and removes special characters automatically."""
    hrdata.columns = (
        hrdata.columns
        .str.lower()
        .str.strip()
        .str.replace('[^a-z0-9_]', '_', regex=True)  # Replaces spaces, hyphens, & () with _
        .str.replace('_+', '_', regex=True)          # Fixes double underscores like __
        .str.strip('_')                              # Removes trailing/leading underscores
    )
    return hrdata

# call the function
hrdata = clean_column_names(hrdata)

#Final Columns
print(f"Cleaned Headings\n",hrdata.columns)

Cleaned Headings
 Index(['employee_id', 'first_name', 'last_name', 'department', 'salary',
       'hire_date', 'age', 'gender', 'performance_score', 'full_time', 'bonus',
       'marital_status', 'education_level', 'work_experience_years',
       'employee_type', 'office_location', 'project_count',
       'last_promotion_year', 'remote_work_status', 'annual_training_hours',
       'manager_feedback_score'],
      dtype='str')


**Step 2: Identify and Remove Duplicates**
- There are 7 duplicated rows in the dataset

In [6]:
# Check duplicate rows
print("Duplicates before:", hrdata.duplicated().sum())

# Remove duplicate rows
hrdata = hrdata.drop_duplicates()

#OR hrdata.drop_duplicates(inplace=True)

# Verify
print("Duplicates after:", hrdata.duplicated().sum())

Duplicates before: 7
Duplicates after: 0


**Remove Missing Values in employee_id**

In [7]:
# Inspected rows with missing employee_id 
missing_id_rows = hrdata[hrdata['employee_id'].isnull()]

# Permanently dropped rows where employee_id is null 
hrdata = hrdata.dropna(subset=["employee_id"])

# Verified that nulls are now 0
print(hrdata["employee_id"].isnull().sum())  # Now correctly returns 0

0


**Step 3. Categorical Columns Cleaning and Standardization**

- Fix Category Misspellings, Typos and missing values

In [8]:
# Selected columns containing text/categorical data

categorical_columns = hrdata.select_dtypes(include=['object', 'string', 'category']).columns

# Displayed the categorical column names
categorical_columns

Index(['employee_id', 'first_name', 'last_name', 'department', 'salary',
       'hire_date', 'age', 'gender', 'performance_score', 'full_time', 'bonus',
       'marital_status', 'education_level', 'work_experience_years',
       'employee_type', 'office_location', 'project_count',
       'last_promotion_year', 'remote_work_status', 'manager_feedback_score'],
      dtype='str')

"first_name","last_name"

In [9]:
# Clean and title case names
hrdata['first_name'] = hrdata['first_name'].str.strip().str.title()
hrdata['last_name'] = hrdata['last_name'].str.strip().str.title()

# Correctly verify column data types
print("first_name column dtype:", hrdata['first_name'].dtype)
print("last_name column dtype:", hrdata['last_name'].dtype)


first_name column dtype: str
last_name column dtype: str


'department'

In [10]:
# Convert to title case and remove leading and trailing spaces
hrdata["department"] = hrdata["department"].str.strip().str.title()

# List unique categories in 'department'
print(hrdata["department"].unique())

# See each category and how many times it appears
print(hrdata["department"].value_counts())

<ArrowStringArray>
[              'Hr',              'H.R',               'It',
            'Sales',       'Operations',        'Info Tech',
          'Finance',        'Marketing',             'Sale',
         'Finanace',  'Human Resources',  'Humna Resources',
   'Human Resource',      'Humman Res.',         'Markting',
 'Information Tech',                nan,        'Operatons',
              'Ops',              'I.T']
Length: 20, dtype: str
department
Finance             150
Hr                  131
It                  129
Operations          120
Sales               119
Marketing           108
H.R                  12
Info Tech            12
Human Resources      12
Human Resource        9
Operatons             8
Ops                   7
Markting              6
Information Tech      6
Sale                  5
Finanace              5
Humman Res.           4
I.T                   4
Humna Resources       3
Name: count, dtype: int64


In [11]:
# Standardize categories only to Finance, Human Resources, Operations, Marketing, Information Tech, Sales, and Unknown
hrdata["department"] = hrdata["department"].replace({
    "Finanace": "Finance", 
    "Hr": "Human Resources", "H.R": "Human Resources", "Humna Resources": "Human Resources", "Human Resource": "Human Resources", "Humman Res.": "Human Resources",
    "Markting": "Marketing",
    "Ops": "Operations", "Operatons": "Operations",
    "Sale": "Sales",
    "I.T": "Information Tech", "Info Tech": "Information Tech", "It": "Information Tech",
    "nan": "Unknown"
})

# Verify the result
print(hrdata["department"].value_counts())

# Check data type
print(hrdata["department"].dtype)

department
Human Resources     171
Finance             155
Information Tech    151
Operations          135
Sales               124
Marketing           114
Name: count, dtype: int64
str


In [12]:
# Check current null count
print("Nulls before:", hrdata['department'].isnull().sum())

# Fill null values in department with 'Unknown'
hrdata['department'] = hrdata['department'].fillna('Unknown')

# Verify null count is now 0
print("Nulls after:", hrdata['department'].isnull().sum())

Nulls before: 12
Nulls after: 0


'gender'

In [13]:
# Convert to tiltle case and remove leading and training spaces
hrdata['gender']=hrdata['gender'].str.strip().str.title()

# List unique categories in 'Gender'
print(hrdata["gender"].unique())

# See each category and how many times it appears
print(hrdata["gender"].value_counts())

<ArrowStringArray>
['Female', 'Male', 'Femle', 'M', nan, 'F', 'Prefer Not Say']
Length: 7, dtype: str
gender
Male              407
Female            403
Femle              11
Prefer Not Say      9
F                   8
M                   4
Name: count, dtype: int64


In [14]:
#  Standardize categories only to  Male,  Female ,Unknown ,Information Tech, Sales,and Unknown
# Replace 'nan', 'Prefer Not Say' to 'Unknown'
hrdata["gender"] = hrdata["gender"].replace({
    "M": "Male", 
    "F": "Female", "Femle": "Female",
    "Prefer Not Say": "Unknown","nan": "Unknown"
    })

# Verify the result
print(hrdata["gender"].value_counts())

#Check data type
print(type('gender'))

gender
Female     422
Male       411
Unknown      9
Name: count, dtype: int64
<class 'str'>


In [15]:
# Check current null count
print("Nulls before:", hrdata['gender'].isnull().sum())

# Fill null values in gender with 'Unknown'
hrdata['gender'] = hrdata['gender'].fillna('Unknown')

# Verify null count is now 0
print("Nulls after:", hrdata['gender'].isnull().sum())

Nulls before: 20
Nulls after: 0


full_time

In [16]:
# Convert to tiltle case and remove leading and training spaces
hrdata['full_time']=hrdata['full_time'].str.strip().str.title()

# Replace Missing Values
hrdata['full_time'] = hrdata['full_time'].fillna('Unknown')

# See each category and how many times it appears
print(hrdata["full_time"].value_counts())

full_time
Yes          429
No           402
N             10
Unknown        9
Y              5
Part-Time      4
Full Time      3
Name: count, dtype: int64


In [17]:
# Standardize attributes
hrdata["full_time"]=hrdata["full_time"].replace({
    "Y":"Yes","Full Time":"Yes",
    "N":"No","Part-Time":"No",
    "nan":"Unknown"
})

# Count category values
print(hrdata["full_time"].value_counts())

#Check data type
print(type('first_time'))

full_time
Yes        437
No         416
Unknown      9
Name: count, dtype: int64
<class 'str'>


In [18]:
# Check current null count
print("Nulls before:", hrdata['full_time'].isnull().sum())

# Address missing values
hrdata['full_time'] = hrdata['full_time'].fillna('Unknown')

# Verify null count is now 0
print("Nulls after:", hrdata['full_time'].isnull().sum())

Nulls before: 0
Nulls after: 0


'marital_status'

In [19]:
#Remove  leading and trailing  spaces and change to title case
#hrdata['marital_status'].str.title().str.strip() # transform only
hrdata['marital_status'] = hrdata['marital_status'].str.strip().str.title() # transform and save

#Identify  unique categories in the column
print(hrdata['marital_status'].unique())

#Count unique categories in the column
hrdata['marital_status'].value_counts()

<ArrowStringArray>
['Widowed', 'Married', 'Single', 'Divorced', nan, 'Widwowed', 'Maried']
Length: 7, dtype: str


marital_status
Married     219
Widowed     214
Single      209
Divorced    199
Widwowed      7
Maried        4
Name: count, dtype: int64

In [20]:
#Standardize inconsistent categories
hrdata['marital_status']=hrdata['marital_status'].replace({
    "Widwowed":"Widowed",
    "maried":"Married",
    "single":"Single",
    "nan":"Unknown"
})

#Check data type
print(type('marital_status'))

<class 'str'>


In [21]:
# Check current null count
print("Nulls before:", hrdata['marital_status'].isnull().sum())

# Address Missing Values
hrdata['marital_status'] = hrdata['marital_status'].fillna('Unknown')

# Verify null count is now 0
print("Nulls after:", hrdata['marital_status'].isnull().sum())

Nulls before: 10
Nulls after: 0


In [22]:
hrdata.columns

Index(['employee_id', 'first_name', 'last_name', 'department', 'salary',
       'hire_date', 'age', 'gender', 'performance_score', 'full_time', 'bonus',
       'marital_status', 'education_level', 'work_experience_years',
       'employee_type', 'office_location', 'project_count',
       'last_promotion_year', 'remote_work_status', 'annual_training_hours',
       'manager_feedback_score'],
      dtype='str')

education_level

In [23]:
#Remove  leading and trailing  spaces and change to title case
hrdata['education_level']=hrdata['education_level'].str.title().str.strip()

#Identify  unique categories in the column
print(hrdata['education_level'].unique())

#Count unique categories in the column
hrdata['education_level'].value_counts()

<ArrowStringArray>
[        'Phd', 'High School', 'Associate'S',  'Bachelor'S',    'Bachelor',
    'High Sch',    'Master'S',     'Masters',           nan,   'Bachelors',
  'Associates',         'Msc']
Length: 12, dtype: str


education_level
Associate'S    178
Bachelor'S     170
Master'S       165
High School    161
Phd            152
High Sch         6
Bachelors        6
Associates       6
Bachelor         5
Masters          5
Msc              2
Name: count, dtype: int64

In [24]:
#Standardize inconsistent categories
hrdata['education_level']=hrdata['education_level'].replace({
    "PHD":"PhD","phd":"PhD",
    "Associates":"Associate's",
    "high school":"High School","High Sch":"High School",
    "Bachelor":"Bachelors","Bachelor's":"Bachelors",
    "Master's":"Masters", "MSc":"Masters",
    "nan":"Unknown"
})

#Check data type
print(type('education_level'))

<class 'str'>


In [25]:
# Check current null count
print("Nulls before:", hrdata['education_level'].isnull().sum())

# Address missing values
hrdata['education_level'] = hrdata['education_level'].fillna('Unknown')

# Verify null count is now 0
print("Nulls after:", hrdata['education_level'].isnull().sum())

Nulls before: 6
Nulls after: 0


'employee_type'

In [26]:
# Convert the column to title case and remove leading and trailing spaces
hrdata['employee_type']=hrdata['employee_type'].str.title().str.strip()

# List usnique categories in the column
print(hrdata['employee_type'].unique())

#Count all unique categories
hrdata['employee_type'].value_counts()


<ArrowStringArray>
[    'Intern',   'Contract',  'Permanent',  'Temporary',      'Inten',
       'Perm', 'Contractor',    'Contrct',          nan]
Length: 9, dtype: str


employee_type
Intern        280
Permanent     273
Contract      270
Temporary       9
Perm            8
Inten           5
Contractor      5
Contrct         5
Name: count, dtype: int64

In [27]:
#Standardize Inconsistent categories

hrdata['employee_type']=hrdata['employee_type'].replace({
    "Inten":"Intern",
    "Perm":"Permanent",
    "Contractor":"Contract","Contrct":"Contract",
    "nana":"Unknown"
})

#Check data type
print(type('employee_type'))

<class 'str'>


In [28]:
# Check current null count
print("Nulls before:", hrdata['employee_type'].isnull().sum())

# Address missing values
hrdata['employee_type'] = hrdata['employee_type'].fillna('Unknown')

# Verify null count is now 0
print("Nulls after:", hrdata['employee_type'].isnull().sum())

Nulls before: 7
Nulls after: 0


office_location

In [29]:
#Remove leading and trailing  spaces and  change to title cae
hrdata['office_location']=hrdata['office_location'].str.title().str.strip()

#Identify unique categories in the column
print(hrdata['office_location'].unique())

#Count unique categories in the column
hrdata['office_location'].value_counts()

<ArrowStringArray>
[       'Nairob',       'Nairobi',        'Berlin',         'Tokyo',
 'San Francisco',        'London',      'New York', 'San Fransisco',
         'Londn',             nan,         'Tokio',            'Sf',
        'Remote',         'Berln']
Length: 14, dtype: str


office_location
San Francisco    143
Nairobi          136
Tokyo            134
London           133
New York         130
Berlin           128
San Fransisco     11
Londn             11
Tokio              7
Nairob             5
Sf                 4
Remote             4
Berln              3
Name: count, dtype: int64

In [30]:
#Standardize inconsistent Categories
hrdata['office_location']=hrdata['office_location'].replace({
    "Nairob":"Nairobi","NAIROBI":"Nairobi",
    "San Fransisco":"San Francisco","SF":"San Francisco",
    "Londn":"London",
    "Tokio":"Tokyo",
    "Berln":"Berlin",
    " nan":"Unknown"
})

#Check data type
print(type('office_location'))

<class 'str'>


In [31]:
# Check current null count
print("Nulls before:", hrdata['office_location'].isnull().sum())

# Address missing values
hrdata['office_location'] = hrdata['office_location'].fillna('Unknown')

# Verify null count is now 0
print("Nulls after:", hrdata['office_location'].isnull().sum())

Nulls before: 13
Nulls after: 0


remote_work_status

In [32]:
#Remove leading and trailing  spaces and  chnage to title cae
hrdata['remote_work_status']=hrdata['remote_work_status'].str.title().str.strip()

#Identify unique categories in the column
print(hrdata['remote_work_status'].unique())

#Count unique categories in the column
hrdata['remote_work_status'].value_counts()

<ArrowStringArray>
[     'On-Site',       'Hybrid', 'Fully Remote',            nan,
       'Remote',      'On Site',       'Onsite',       'Hybird']
Length: 8, dtype: str


remote_work_status
On-Site         286
Hybrid          285
Fully Remote    254
Onsite            9
Remote            8
On Site           5
Hybird            4
Name: count, dtype: int64

In [33]:
#Standardize inconsistent Categories
hrdata['remote_work_status']=hrdata['remote_work_status'].replace({
    "On site":"On-Site","on-site":"On-Site","Onsite":"On-Site",
    "Fully Remote":"Remote","fully remote":"Remote",
    "Hybird":"Hybrid","hybrid":"Hybrid",
    "nan":"Unknown"
})

#Check data type
print(type('remote_work_status'))

<class 'str'>


In [34]:
# Check current null count
print("Nulls before:", hrdata['remote_work_status'].isnull().sum())

# Address missing values
hrdata['remote_work_status'] = hrdata['remote_work_status'].fillna('Unknown')

# Verify null count is now 0
print("Nulls after:", hrdata['remote_work_status'].isnull().sum())

Nulls before: 11
Nulls after: 0


**Step 4. Numerical Columns Cleaning and Standardization**

- Fix  word numbers,nan,N/A,'', & Data types

**Identify All Numeric Columns in The Dataset**

- Columns like age, bonus, project_count, and manager_feedback_score are numbers.
-  They are still trapped as text (object or str types)  because they contain  dirty characters (like words, spaces, typos, or NaN labels).

In [35]:
# Select and display the names of all true numerical columns
numeric_columns = hrdata.select_dtypes(include=['number']).columns
list(numeric_columns)


['annual_training_hours']

In [36]:
# List of all columns and their data types
hrdata.dtypes


employee_id                object
first_name                    str
last_name                     str
department                    str
salary                     object
hire_date                  object
age                        object
gender                        str
performance_score          object
full_time                     str
bonus                      object
marital_status                str
education_level               str
work_experience_years      object
employee_type                 str
office_location               str
project_count              object
last_promotion_year        object
remote_work_status            str
annual_training_hours     float64
manager_feedback_score     object
dtype: object

**Column By Column Clean Up**

Salary

In [37]:
# Count how many rows have missing or blank salary values
missing_salaries = hrdata['salary'].isnull().sum()

print(f"Number of missing values in Salary: {missing_salaries}")

# List out every unique variation present in the salary column
unique_salaries = hrdata['salary'].unique()

print(unique_salaries[:10]) #Output the first 10 rows

Number of missing values in Salary: 9
[74291 93350 112020 34646 69713 '97936' 74331 52185 114263 '82107']


In [38]:
# Converted salary values to strings, then used regex to remove:
# KES, $, commas, and whitespace, leaving only the numeric value.
#
# Regex: r'KES|\$|,|\s'
# - KES = removes "KES"
# - \$  = removes the "$" symbol
# - ,   = removes commas
# - \s  = removes spaces/whitespace
# - |   = means "OR"
# - regex=True = tells Pandas to treat the pattern as a regex

hrdata['salary'] = hrdata['salary'].astype(str).str.replace(r'KES|\$|,|\s', '', regex=True)


# Converted the clean text back into floats 
hrdata['salary'] = pd.to_numeric(hrdata['salary'], errors='coerce')

# Standardized unique values
print("Missing values now:", hrdata['salary'].isnull().sum())
print("\nFirst 10 clean numbers:\n", hrdata['salary'].dropna().unique()[:10])


Missing values now: 9

First 10 clean numbers:
 [ 74291.  93350. 112020.  34646.  69713.  97936.  74331.  52185. 114263.
  82107.]


In [39]:
# Calculated the median salary 
median_salary = hrdata['salary'].median()
print(f"Median salary is: {median_salary}\n")

# Filled the 9 missing spaces with the median value
hrdata['salary'] = hrdata['salary'].fillna(median_salary)

# confirmed that there are no remaining missing values
print("Missing values remaining in Salary:", hrdata['salary'].isnull().sum())

# Changed data type back to numeric
hrdata['salary'] = pd.to_numeric(hrdata['salary'], errors='coerce')



Median salary is: 76008.0

Missing values remaining in Salary: 0


hire_date

- To fix this, we use pd.to_datetime().

In [40]:
# Changed the column to a date format
hrdata['hire_date'] = pd.to_datetime(hrdata['hire_date'], errors='coerce')

# Checked for missing and broken dates 
missing_dates = hrdata['hire_date'].isnull().sum()
print(f"Number of broken or missing dates: {missing_dates}")

# Retained invalid/missing dates as NaT instead of replacing  missing values
hrdata['hire_date'] = pd.to_datetime(hrdata['hire_date'], errors='coerce')


# Confirmation
print("\nSample of clean dates:")
print(hrdata['hire_date'].dropna().head())


Number of broken or missing dates: 44

Sample of clean dates:
0   2002-09-14
1   2008-11-12
2   2017-07-13
3   2014-07-27
4   2010-07-12
Name: hire_date, dtype: datetime64[us]


age  

In [41]:
# Converted to string, lowercase it, and change written words to numbers(thirty)
hrdata['age'] = hrdata['age'].astype(str).str.lower().str.strip()
hrdata['age'] = hrdata['age'].str.replace('thirty', '30')

# Extracted only numbers 
hrdata['age'] = hrdata['age'].str.extract(r'(\d+\.?\d*)')


# Changed to  numerical float type 
hrdata['age'] = pd.to_numeric(hrdata['age'], errors='coerce')

# Filled all missing values with median age
median_age = hrdata['age'].median()
hrdata['age'] = hrdata['age'].fillna(median_age).astype(int) # Convert to integer at the end

# 6. Verify our work
print(f"median_age: {int(company_median_age)}")
print("Remaining missing values:", hrdata['age'].isnull().sum())



NameError: name 'company_median_age' is not defined

performance_score

In [ ]:
# Changed to numeric datatype
hrdata['performance_score'] = pd.to_numeric(hrdata['performance_score'], errors='coerce')

# Filled missing values with  column median
median_score = hrdata['performance_score'].median()
hrdata['performance_score'] = hrdata['performance_score'].fillna(median_score).astype(int)

print("Missing values remaining:", hrdata['performance_score'].isnull().sum())


Missing values remaining: 0


bonus

In [ ]:
# Stripped out currency tags (KES, $) commas, and spaces using a regular expression
hrdata['bonus'] = hrdata['bonus'].astype(str).str.replace(r'[\$,KRES\s-]', '', regex=True)

# Changed to numeric data type

hrdata['bonus'] = pd.to_numeric(hrdata['bonus'], errors='coerce')

# Fill all missing values/NaNs with 0 (Assuming no data means $0 bonus)
hrdata['bonus'] = hrdata['bonus'].fillna(0)


work_experience_years

In [ ]:
# Converted to string, lowercase it, and remove extra spaces
hrdata['work_experience_years'] = hrdata['work_experience_years'].astype(str).str.lower().str.strip()

# Extracted only numbers and decimals
hrdata['work_experience_years'] = hrdata['work_experience_years'].str.extract(r'(\d+\.?\d*)')

# Chnaged numeric data type
hrdata['work_experience_years'] = pd.to_numeric(hrdata['work_experience_years'], errors='coerce')

# Handled negative numbers (I chnaged any value below 0 as a missing value (NaN))
hrdata.loc[hrdata['work_experience_years'] < 0, 'work_experience_years'] = np.nan

# Filled missing values using column median
median_experience = hrdata['work_experience_years'].median()
hrdata['work_experience_years'] = hrdata['work_experience_years'].fillna(median_experience).astype(int)


project_count

In [ ]:
# Replace({'ten': '10'})
hrdata['project_count'] = hrdata['project_count'].astype(str).str.lower().str.strip().replace({'ten': '10'})

# Changed data type numeric
hrdata['project_count'] = pd.to_numeric(hrdata['project_count'], errors='coerce')

#  Replaces missing values with column median
hrdata['project_count'] = hrdata['project_count'].fillna(hrdata['project_count'].median()).astype(int)


last_promotion_year 

In [ ]:
# Changed to numeric data type 
hrdata['last_promotion_year'] = pd.to_numeric(hrdata['last_promotion_year'], errors='coerce')

# Replaced the new NaN blanks with 0.(0 meant not yet promoted)
hrdata['last_promotion_year'] = hrdata['last_promotion_year'].fillna(0).astype(int)


annual_training_hours 

In [ ]:
# Calculated column median
training_median = hrdata['annual_training_hours'].median()

# Replaced missing values with column median
hrdata['annual_training_hours'] = hrdata['annual_training_hours'].fillna(training_median)

# Changed to int data type
hrdata['annual_training_hours'] = hrdata['annual_training_hours'].astype(int)


manager_feedback_score 

In [ ]:
# Created a dictionary to convert feedback words into standard numbers on a 1-5 scale.
score_mapping = {'Excellent': 5, 'Good': 4, 'Poor': 1, 'N/A': np.nan, 'None': np.nan}

# Replaced "Good" and  "Excellent" with their numeric equivalents.
hrdata['manager_feedback_score'] = hrdata['manager_feedback_score'].astype(str).str.strip().replace(score_mapping)

# Changed to numeric data type
hrdata['manager_feedback_score'] = pd.to_numeric(hrdata['manager_feedback_score'], errors='coerce')

# Replaced  missing values with median
feedback_median = hrdata['manager_feedback_score'].median()
hrdata['manager_feedback_score'] = hrdata['manager_feedback_score'].fillna(feedback_median).round(1)


**Step 5. Final Quality Report**

In [ ]:
quality_report = pd.DataFrame({
    "Column": hrdata.columns,
    "Data_Type": hrdata.dtypes.astype(str),
    "Total_Rows": len(hrdata),
    "Missing_Values": hrdata.isna().sum().values,
    "Missing_Percentage": (hrdata.isna().mean() * 100).round(2).values,
    "Unique_Values": hrdata.nunique().values,
})

quality_report


,Column,Data_Type,Total_Rows,Missing_Values,Missing_Percentage,Unique_Values
employee_id,employee_id,object,862,0,0.00,852
first_name,first_name,str,862,11,1.28,91
last_name,last_name,str,862,9,1.04,60
department,department,str,862,0,0.00,7
salary,salary,float64,862,0,0.00,846
hire_date,hire_date,datetime64[us],862,44,5.10,760
age,age,int64,862,0,0.00,46
gender,gender,str,862,0,0.00,3
performance_score,performance_score,int64,862,0,0.00,11
full_time,full_time,str,862,0,0.00,3


**Step 6.Data Export**

In [ ]:
# Save cleaned DataFrame to a new Excel file
hrdata.to_excel('HR_Cleaned_Data.xlsx', index=False)